In [1]:
import time
import random
import pandas as pd

from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.service import Service

from webdriver_manager.chrome import ChromeDriverManager



districts = [
    "Bokaro",
    "Chatra",
    "Deoghar",
    "Dhanbad",
    "Dumka",
    "East Singhbhum",
    "Garhwa",
    "Giridih",
    "Godda",
    "Gumla",
    "Hazaribagh",
    "Jamtara",
    "Khunti",
    "Koderma",
    "Latehar",
    "Lohardaga",
    "Pakur",
    "Palamu",
    "Ramgarh",
    "Ranchi",
    "Sahebganj",
    "Seraikela Kharsawan",
    "Simdega",
    "West Singhbhum"
]


search_types = [
    "software companies",
    "IT companies"
]


options = webdriver.ChromeOptions()

options.add_argument("--start-maximized")

options.add_argument(
    "--disable-blink-features=AutomationControlled"
)

options.add_argument("--disable-notifications")


driver = webdriver.Chrome(

    service=Service(
        ChromeDriverManager().install()
    ),

    options=options
)



driver.get("https://www.google.com/maps")

print("Opening Google Maps...")

time.sleep(15)

print("Google Maps Loaded")



all_data = []

visited_links = set()



def get_search_box():

    selectors = [

        '//input[@id="searchboxinput"]',
        '//input[contains(@placeholder,"Search")]',
        '//input[contains(@aria-label,"Search")]',
        '//input'

    ]

    for selector in selectors:

        try:

            box = driver.find_element(
                By.XPATH,
                selector
            )

            return box

        except:
            pass

    return None



def extract_school_data():

    try:

        try:

            name = driver.find_element(
                By.TAG_NAME,
                "h1"
            ).text

        except:

            name = ""


        try:

            address = driver.find_element(
                By.XPATH,
                '//button[contains(@data-item-id,"address")]'
            ).text

        except:

            address = ""


        phone = ""

        try:

            phone_xpaths = [

                '//button[contains(@data-item-id,"phone")]',

                '//a[contains(@href,"tel:")]',

                '//button[contains(@aria-label,"Phone")]',

                '//button[contains(text(),"+91")]',
                '//div[contains(text(),"+91")]',
                '//span[contains(text(),"+91")]'

            ]

            for xpath in phone_xpaths:

                try:

                    elements = driver.find_elements(
                        By.XPATH,
                        xpath
                    )

                    for element in elements:

                        text = element.text.strip()

                        href = (
                            element.get_attribute("href")
                            or ""
                        )

                        if "tel:" in href:

                            phone = href.replace(
                                "tel:",
                                ""
                            ).strip()

                        elif text:

                            phone = text

                        phone = (

                            phone.replace("", "")
                            .replace(" ", "")
                            .replace("-", "")
                            .replace("(", "")
                            .replace(")", "")
                            .replace("\n", "")
                            .strip()

                        )

  
                        digits = ''.join(
                            filter(str.isdigit, phone)
                        )

                        if len(digits) >= 10:

                            break

                    if phone:
                        break

                except:
                    pass

        except Exception as e:

            print(f"Phone Error: {e}")



        try:

            website = driver.find_element(
                By.XPATH,
                '//a[contains(@data-item-id,"authority")]'
            ).get_attribute("href")

        except:

            website = ""

        # RATING

        try:

            rating = driver.find_element(
                By.XPATH,
                '//div[@role="img"]'
            ).get_attribute("aria-label")

        except:

            rating = ""

        return {

            "Company Name": name,
            "Address": address,
            "Phone": phone,
            "Website": website,
            "Rating": rating,
            "Google Maps URL": driver.current_url

        }

    except:

        return None



for district in districts:

    for search_type in search_types:

        try:

            query = (
                f"{search_type} "
                f"in {district} Jharkhand"
            )

            print(f"\nSearching: {query}")



            search_box = get_search_box()

            if not search_box:

                print("Search Box Not Found")

                continue

            ActionChains(driver).move_to_element(
                search_box
            ).click().perform()

            time.sleep(1)

            search_box.send_keys(
                Keys.CONTROL + "a"
            )

            time.sleep(1)

            search_box.send_keys(Keys.DELETE)

            time.sleep(1)



            for char in query:

                search_box.send_keys(char)

                time.sleep(0.03)

            search_box.send_keys(Keys.ENTER)

            print("Search Submitted")

            time.sleep(8)


            try:

                scrollable_div = driver.find_element(
                    By.XPATH,
                    '//div[@role="feed"]'
                )

                previous_count = 0

                stagnant = 0

                for i in range(100):

                    driver.execute_script(
                        """
                        arguments[0].scrollTop =
                        arguments[0].scrollHeight
                        """,
                        scrollable_div
                    )

                    time.sleep(4)

                    listings = driver.find_elements(
                        By.XPATH,
                        '//a[contains(@href,"/maps/place/")]'
                    )

                    current_count = len(listings)

                    print(
                        f"{district} | "
                        f"{search_type} | "
                        f"Scroll {i+1} -> "
                        f"{current_count}"
                    )

                    if current_count == previous_count:

                        stagnant += 1

                    else:

                        stagnant = 0

                    if stagnant >= 4:

                        break

                    previous_count = current_count

            except Exception as e:

                print(f"Scrolling Error: {e}")

            listings = driver.find_elements(
                By.XPATH,
                '//a[contains(@href,"/maps/place/")]'
            )

            links = []

            for listing in listings:

                try:

                    link = listing.get_attribute("href")

                    if (
                        link
                        and link not in visited_links
                    ):

                        links.append(link)

                        visited_links.add(link)

                except:
                    pass

            print(f"Collected {len(links)} links")

     
            for index, link in enumerate(links):

                try:

                    print(
                        f"{district} | "
                        f"{index+1}/{len(links)}"
                    )

                    driver.get(link)

                    time.sleep(
                        random.uniform(2, 3)
                    )

                    data = extract_school_data()

                    if data:

                        data["District"] = district

                        data["Search Type"] = search_type

                        all_data.append(data)

                        print(data["Company Name"])

                        print(
                            f"Phone: {data['Phone']}"
                        )

        

                    if len(all_data) % 100 == 0:

                        backup_df = pd.DataFrame(
                            all_data
                        )

                        backup_df.drop_duplicates(
                            subset=[
                                "Company Name",
                                "Google Maps URL"
                            ],
                            inplace=True
                        )

                        backup_df.to_csv(
                            "jharkhand_companies_backup.csv",
                            index=False
                        )

                        print("Backup Saved")

                except Exception as e:

                    print(
                        f"Company Error: {e}"
                    )

        except Exception as e:

            print(
                f"District Error: {e}"
            )


df = pd.DataFrame(all_data)

df.drop_duplicates(

    subset=[
        "Company Name",
        "Google Maps URL"
    ],

    inplace=True
)

filename = "jharkhand_companies.csv"

df.to_csv(
    filename,
    index=False
)

print("\nExtraction Completed")

print(
    f"Total Companies Extracted: {len(df)}"
)

print(f"\nSaved File: {filename}")

driver.quit()

Opening Google Maps...
Google Maps Loaded

Searching: software companies in Bokaro Jharkhand
Search Submitted
Bokaro | software companies | Scroll 1 -> 14
Bokaro | software companies | Scroll 2 -> 20
Bokaro | software companies | Scroll 3 -> 27
Bokaro | software companies | Scroll 4 -> 34
Bokaro | software companies | Scroll 5 -> 40
Bokaro | software companies | Scroll 6 -> 47
Bokaro | software companies | Scroll 7 -> 54
Bokaro | software companies | Scroll 8 -> 60
Bokaro | software companies | Scroll 9 -> 67
Bokaro | software companies | Scroll 10 -> 74
Bokaro | software companies | Scroll 11 -> 80
Bokaro | software companies | Scroll 12 -> 87
Bokaro | software companies | Scroll 13 -> 94
Bokaro | software companies | Scroll 14 -> 96
Bokaro | software companies | Scroll 15 -> 96
Bokaro | software companies | Scroll 16 -> 96
Bokaro | software companies | Scroll 17 -> 96
Bokaro | software companies | Scroll 18 -> 96
Collected 96 links
Bokaro | 1/96
Wipenex IT Private Limited
Phone: 0943